<a href="https://colab.research.google.com/github/manubastidas/programacionCientifica/blob/eval2-ZapataGonzalez/Evaluacion2/ZapataGonzalez/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!sudo apt-get update
!sudo apt-get install texlive-latex-extra texlive-fonts-recommended dvipng cm-super

import numpy as np


from scipy.special import comb
from scipy.interpolate import CubicSpline, make_interp_spline

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "font.size": 14,

    # Ejes y Ticks
    "axes.labelsize": 16,
    "axes.titlesize": 18,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.direction": "in",
    "ytick.direction": "in",

    # Grid                          # ← bug 1: sintaxis mezclada
    "grid.color"    : "gray",       # todo debe ir como claves del dict
    "grid.linewidth": 0.3,
    "grid.alpha"    : 0.3,
    "grid.linestyle": "--",

    # Estética
    "figure.dpi"        : 120,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "savefig.bbox"      : "tight",
    "savefig.dpi"       : 300,
})

print('Configuración OK')

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,082 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.4 MB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,258 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:14 ht

In [ ]:
!python generar_enunciado.py 3951
!python generar_datos.py 3951

Enunciado generado para cédula 3951: enunciado_3951.md
Los parámetros: K=25, iteraciones=3000, η=0.02, n=12/régimen, régimen foco=0, compresión 85%
Datos generados para cédula 3951: X=(36, 256), y=(36,), t=(256,)
Guardado en datos_3951.npz


In [1]:
!pip install --upgrade jax jaxlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 10.2 MB/s eta 0:00:00
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.7.2
    Uninstalling jaxlib-0.7.2:
      Successfully uninstalled jaxlib-0.7.2
  Attempting uninstall: jax
    Found existing installation: jax 0.7.2
    Uninstalling jax-0.7.2:
      Successfully uninstalled jax-0.7.2


**Capa Fourier**
El siguiente código contiene los resultados pedidos en la parte "capa fourier (15 pts)" de la rúbrica. (función de la capa de fourier, comparación con np.fft (evidencias matemáticas) y reporte de energía)

In [8]:
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap
import numpy as np
import matplotlib.pyplot as plt

# 1. Se definen los parámetros según lo generado anteriormente
K = 25              # Número de frecuencias
hidden_dim = 2 * K  # 2K = 50 neuronas ocultas
n_features = 256    # Tamaño de la señal (t)
eta = 0.02          # Learning rate
iteraciones = 3000  # Número de iteraciones

# Se cargan los datos generados con mi ID
datos = np.load("datos_3951.npz")
X_train = jnp.array(datos['X'])
y_train = np.array(datos['y'])
t = jnp.array(datos['t'])

# CAPA FOURIER
def construir_capa_fourier(n_features=256, K=25):
    """
    Construye la matriz W1 inicializada con la base de Fourier.
    Dimensión resultante: (2K, n_features) -> (50, 256)
    """
    t_arr = jnp.linspace(0, 1, n_features, endpoint=False)
    W1_list = []

    # Filas alternadas o agrupadas de cosenos y senos para k=1,...,K
    for k in range(1, K + 1):
        W1_list.append(jnp.cos(2 * jnp.pi * k * t_arr) * jnp.sqrt(2.0 / n_features))
        W1_list.append(jnp.sin(2 * jnp.pi * k * t_arr) * jnp.sqrt(2.0 / n_features))

    return jnp.stack(W1_list, axis=0)

W1_fourier_init = construir_capa_fourier(n_features, K) #Capa de Fourier

# ARQUITECTURA DE LA RED NEURONAL
# Función Forward: x_hat = W2 @ tanh(W1 @ x)
def forward(params, x):
    phi = jnp.tanh(jnp.dot(params['W1'], x))
    x_hat = jnp.dot(params['W2'], phi)
    return x_hat

# Vectorizamos para procesar todas las señales (36, 256) en paralelo
forward_batch = vmap(forward, in_axes=(None, 0))

# Función de Pérdida (MSE)
@jit
def loss_fn(params, X):
    X_hat = forward_batch(params, X)
    return jnp.mean((X_hat - X) ** 2)

# Gradiente Descendente para actualizar pesos
@jit
def update(params, X, lr):
    grads = grad(loss_fn)(params, X)
    updated_params = {
        'W1': params['W1'] - lr * grads['W1'],
        'W2': params['W2'] - lr * grads['W2']
    }
    return updated_params, grads




# Reporte de Energía Media (Con 3 decimales exactos)
energia_media = float((X_train**2).mean())
print(f"ENERGÍA MEDIA")
print(f"La energía media del dataset es: {energia_media:.3f}\n")

# Evidencia de Inicialización, comparando con np.fft

proyeccion_fourier = jnp.dot(W1_fourier_init, X_train[0]) # Proyectamos la primera señal usando nuestra capa de Fourier

fft_np = np.fft.rfft(X_train[0]) # Calculamos la FFT tradicional de NumPy para verificar

print("EVIDENCIA MATEMÁTICA DE FOURIER")
print(f"Magnitud de las primeras 2 componentes (Capa W1): {jnp.abs(proyeccion_fourier[:2])}")
print(f"Magnitud de las primeras 2 componentes (np.fft):  {np.abs(fft_np[1:3]) / jnp.sqrt(128)}") # Ajuste de escala

ENERGÍA MEDIA
La energía media del dataset es: 0.621

EVIDENCIA MATEMÁTICA DE FOURIER
Magnitud de las primeras 2 componentes (Capa W1): [4.8337393  0.11186668]
Magnitud de las primeras 2 componentes (np.fft):  [4.835033   0.01968224]
